# 改进布林带趋势增强策略

## 论文参考

- **标题**: Trend-Enhanced Improved Bollinger Bands
- **作者**: L. Gong
- **年份**: 2024
- **关键结果**: 年化 123.9%, 夏普比率 3.26

### 摘要

本文提出了对经典布林带策略的四项关键改进:
1. 使用 EMA 替代 SMA 作为中轨
2. 使用 ATR 自适应调整带宽
3. 加入 ADX 趋势过滤器
4. 使用 MACD + CCI 多重确认信号

纯规则策略，无机器学习。使用 510300 ETF 日线数据回测。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### 改进布林带

**1. EMA 中轨 (替代 SMA)**

$$\text{Middle}_t = \text{EMA}(\text{Close}, N)$$

EMA 对近期价格更敏感，反应更快。

**2. ATR 自适应带宽**

$$\text{Upper}_t = \text{Middle}_t + k \cdot \text{ATR}(N)$$
$$\text{Lower}_t = \text{Middle}_t - k \cdot \text{ATR}(N)$$

ATR 替代标准差，波动大时带宽自动扩大。

**3. ADX 趋势过滤**

$$\text{ADX} > 25 \implies \text{趋势市场 (执行趋势跟踪信号)}$$
$$\text{ADX} < 20 \implies \text{震荡市场 (执行均值回归信号)}$$

**4. MACD + CCI 确认**

- **MACD**: 柱状图方向确认趋势
- **CCI > 100**: 超买确认; **CCI < -100**: 超卖确认

**5. 交易规则**

趋势模式 (ADX > 25):
- 买入: 价格上穿上轨 + MACD 柱状图 > 0
- 卖出: 价格下穿中轨 或 CCI < -100

震荡模式 (ADX < 20):
- 买入: 价格触及下轨 + CCI < -100 (超卖反弹)
- 卖出: 价格触及上轨 + CCI > 100 (超买回落)

In [ ]:
# ============================================================
# Cell 3: 动画展示 - 交互式改进布林带 (plotly)
# ============================================================
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)
n = 200
# 模拟价格
price = 4.0 + np.cumsum(np.random.randn(n) * 0.03)
price = np.maximum(price, 2.0)
high_demo = price + np.abs(np.random.randn(n) * 0.02)
low_demo = price - np.abs(np.random.randn(n) * 0.02)

# EMA布林带
close_s = pd.Series(price)
ema_mid = close_s.ewm(span=20).mean()
# ATR
tr = pd.concat([
    pd.Series(high_demo) - pd.Series(low_demo),
    (pd.Series(high_demo) - close_s.shift(1)).abs(),
    (pd.Series(low_demo) - close_s.shift(1)).abs()
], axis=1).max(axis=1)
atr_val = tr.rolling(14).mean()
upper = ema_mid + 2.0 * atr_val
lower = ema_mid - 2.0 * atr_val

# 创建动画帧
frames = []
step = 5
for t in range(40, n, step):
    frames.append(go.Frame(
        data=[
            go.Scatter(x=list(range(t)), y=price[:t], mode='lines',
                       name='Price', line=dict(color='#333', width=1.5)),
            go.Scatter(x=list(range(t)), y=ema_mid[:t].values, mode='lines',
                       name='EMA Middle', line=dict(color='#2196F3', width=2)),
            go.Scatter(x=list(range(t)), y=upper[:t].values, mode='lines',
                       name='Upper (EMA+2ATR)', line=dict(color='#F44336', dash='dash')),
            go.Scatter(x=list(range(t)), y=lower[:t].values, mode='lines',
                       name='Lower (EMA-2ATR)', line=dict(color='#4CAF50', dash='dash')),
            go.Scatter(x=list(range(t)) + list(range(t))[::-1],
                       y=list(upper[:t].values) + list(lower[:t].values)[::-1],
                       fill='toself', fillcolor='rgba(33,150,243,0.08)',
                       line=dict(color='rgba(0,0,0,0)'), name='Band'),
        ],
        name=f't={t}'
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title=dict(text='改进布林带: EMA中轨 + ATR自适应带宽'),
        xaxis_title='时间', yaxis_title='价格',
        height=500, width=950,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='\u25b6 播放', method='animate',
                 args=[None, {'frame': {'duration': 150}, 'transition': {'duration': 80}}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
        )],
    )
)
fig.show()

In [ ]:
# ============================================================
# Cell 4: 环境设置与导入
# ============================================================
import sys
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
np.random.seed(42)

sys.path.insert(0, '..')
from shared.data_fetcher import get_etf_daily, get_index_daily
from shared.backtest_engine import Backtester
from shared.factors import ema, atr, adx, macd, cci
from shared.visualization import (
    plot_equity_curve, plot_drawdown, plot_signals, plot_metrics_table
)
from shared.constants import DEFAULT_START, DEFAULT_END, ETF_CSI300

print('所有模块导入成功')

In [ ]:
# ============================================================
# Cell 5: 数据获取
# ============================================================
try:
    df = get_etf_daily(ETF_CSI300, DEFAULT_START, DEFAULT_END)
    print(f'ETF 510300 数据: {len(df)} 条')
except Exception as e:
    print(f'数据获取异常: {e}, 使用模拟数据')
    dates = pd.bdate_range(DEFAULT_START, DEFAULT_END)
    n = len(dates)
    np.random.seed(42)
    price = 4.5 + np.cumsum(np.random.randn(n) * 0.015)
    price = np.maximum(price, 2.0)
    df = pd.DataFrame({
        'open': price * (1 + np.random.randn(n) * 0.003),
        'close': price,
        'high': price * (1 + np.abs(np.random.randn(n) * 0.008)),
        'low': price * (1 - np.abs(np.random.randn(n) * 0.008)),
        'volume': np.random.randint(50000, 2000000, n).astype(float),
    }, index=dates)

print(f'数据范围: {df.index[0].strftime("%Y-%m-%d")} ~ {df.index[-1].strftime("%Y-%m-%d")}')
print(f'收盘价范围: {df["close"].min():.2f} ~ {df["close"].max():.2f}')

In [ ]:
# ============================================================
# Cell 6: 改进布林带指标计算
# ============================================================

# 参数设置
BB_WINDOW = 20
ATR_WINDOW = 14
ADX_WINDOW = 14
ATR_MULT = 2.0
ADX_TREND_THRESH = 25
ADX_RANGE_THRESH = 20

# 1. EMA 中轨
df['ema_middle'] = ema(df['close'], BB_WINDOW)

# 2. ATR 自适应带宽
df['atr'] = atr(df['high'], df['low'], df['close'], ATR_WINDOW)
df['bb_upper'] = df['ema_middle'] + ATR_MULT * df['atr']
df['bb_lower'] = df['ema_middle'] - ATR_MULT * df['atr']

# 3. ADX 趋势指标
df['adx'] = adx(df['high'], df['low'], df['close'], ADX_WINDOW)

# 4. MACD
macd_df = macd(df['close'])
df['macd_hist'] = macd_df['histogram']
df['macd_line'] = macd_df['macd']
df['macd_signal'] = macd_df['signal']

# 5. CCI
df['cci'] = cci(df['high'], df['low'], df['close'], 20)

# 传统SMA布林带 (对照组)
from shared.factors import sma
df['sma_middle'] = sma(df['close'], BB_WINDOW)
std_val = df['close'].rolling(BB_WINDOW).std()
df['sma_upper'] = df['sma_middle'] + 2 * std_val
df['sma_lower'] = df['sma_middle'] - 2 * std_val

print('指标计算完成')
print(f'ADX 统计: mean={df["adx"].mean():.1f}, 趋势市场占比={((df["adx"] > ADX_TREND_THRESH).mean()):.1%}')
print(f'CCI 统计: mean={df["cci"].mean():.1f}, 超买占比={((df["cci"] > 100).mean()):.1%}, 超卖占比={((df["cci"] < -100).mean()):.1%}')

In [ ]:
# ============================================================
# Cell 7: 信号生成 (改进布林带 + 经典布林带)
# ============================================================

def improved_bollinger_signals(df):
    """改进布林带信号: EMA + ATR + ADX + MACD/CCI"""
    n = len(df)
    signals = pd.Series(0, index=df.index)
    position = 0  # 0=空仓, 1=持仓

    for i in range(1, n):
        close = df['close'].iloc[i]
        upper = df['bb_upper'].iloc[i]
        lower = df['bb_lower'].iloc[i]
        middle = df['ema_middle'].iloc[i]
        adx_val = df['adx'].iloc[i]
        macd_h = df['macd_hist'].iloc[i]
        cci_val = df['cci'].iloc[i]

        if pd.isna(adx_val) or pd.isna(upper):
            signals.iloc[i] = position
            continue

        if adx_val > ADX_TREND_THRESH:
            # === 趋势模式 ===
            if position == 0:
                # 买入: 价格上穿上轨 + MACD > 0
                if close > upper and macd_h > 0:
                    position = 1
                # 买入: 价格在中轨上方 + MACD金叉
                elif close > middle and macd_h > 0 and df['macd_hist'].iloc[i-1] <= 0:
                    position = 1
            else:
                # 卖出: 价格下穿中轨 或 CCI < -100
                if close < middle or cci_val < -100:
                    position = 0

        elif adx_val < ADX_RANGE_THRESH:
            # === 震荡模式 ===
            if position == 0:
                # 买入: 价格触及下轨 + CCI超卖
                if close <= lower and cci_val < -100:
                    position = 1
            else:
                # 卖出: 价格触及上轨 + CCI超买
                if close >= upper and cci_val > 100:
                    position = 0
                # 卖出: 价格回到中轨
                elif close >= middle and cci_val > 0:
                    position = 0
        else:
            # 过渡区间: 维持当前持仓
            pass

        signals.iloc[i] = position

    return signals


def classic_bollinger_signals(df):
    """经典布林带信号 (对照组)"""
    signals = pd.Series(0, index=df.index)
    position = 0

    for i in range(1, len(df)):
        close = df['close'].iloc[i]
        upper = df['sma_upper'].iloc[i]
        lower = df['sma_lower'].iloc[i]
        middle = df['sma_middle'].iloc[i]

        if pd.isna(upper):
            signals.iloc[i] = position
            continue

        if position == 0 and close <= lower:
            position = 1  # 触及下轨买入
        elif position == 1 and close >= upper:
            position = 0  # 触及上轨卖出

        signals.iloc[i] = position

    return signals


# 生成信号
improved_signals = improved_bollinger_signals(df)
classic_signals = classic_bollinger_signals(df)

print(f'改进策略持仓天数: {(improved_signals == 1).sum()} / {len(improved_signals)}')
print(f'经典策略持仓天数: {(classic_signals == 1).sum()} / {len(classic_signals)}')

In [ ]:
# ============================================================
# Cell 8: 回测对比
# ============================================================

try:
    bench = get_index_daily('sh000300', DEFAULT_START, DEFAULT_END)
    bench_prices = bench['close']
except:
    bench_prices = None

backtester = Backtester(initial_capital=1_000_000, t_plus_1=True)

# 改进布林带
improved_result = backtester.run(df['close'], improved_signals, bench_prices)
print('=== 改进布林带策略 ===')
for k, v in improved_result['metrics'].items():
    print(f'  {k}: {v}')

# 经典布林带
classic_result = backtester.run(df['close'], classic_signals, bench_prices)
print('\n=== 经典布林带策略 ===')
for k, v in classic_result['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 9: 绩效可视化
# ============================================================
import matplotlib.pyplot as plt
from shared.visualization import plot_cumulative_comparison

# 1. 策略对比
strategies = {
    '改进布林带': improved_result['returns'],
    '经典布林带': classic_result['returns'],
}
plot_cumulative_comparison(strategies, title='改进 vs 经典布林带 - 策略对比')

# 2. 改进策略收益曲线
bench_eq = None
if improved_result.get('benchmark_returns') is not None:
    bench_eq = (1 + improved_result['benchmark_returns']).cumprod() * improved_result['equity_curve'].iloc[0]
plot_equity_curve(improved_result['equity_curve'], bench_eq,
                  title='改进布林带 - 累计收益')

# 3. 交易信号图
if len(improved_result.get('trades', pd.DataFrame())) > 0:
    trades = improved_result['trades']
    buy_d = trades[trades['action'] == '买入']['date'].tolist()
    sell_d = trades[trades['action'] == '卖出']['date'].tolist()
    plot_signals(df['close'], buy_d, sell_d, title='改进布林带 - 交易信号')

# 4. 回撤
plot_drawdown(improved_result['equity_curve'], title='改进布林带 - 回撤')

# 5. ADX 分布 + 市场状态
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(df.index, df['adx'], color='#9C27B0', linewidth=1)
ax1.axhline(y=25, color='red', linestyle='--', alpha=0.5, label='趋势阈值 (25)')
ax1.axhline(y=20, color='blue', linestyle='--', alpha=0.5, label='震荡阈值 (20)')
ax1.fill_between(df.index, 25, df['adx'], where=df['adx'] > 25,
                 color='red', alpha=0.1, label='趋势市场')
ax1.fill_between(df.index, df['adx'], 20, where=df['adx'] < 20,
                 color='blue', alpha=0.1, label='震荡市场')
ax1.set_title('ADX 趋势/震荡判别')
ax1.set_ylabel('ADX')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# CCI 分布
ax2.hist(df['cci'].dropna(), bins=60, color='#FF9800', alpha=0.7, density=True)
ax2.axvline(x=100, color='red', linestyle='--', label='超买 (100)')
ax2.axvline(x=-100, color='green', linestyle='--', label='超卖 (-100)')
ax2.set_title('CCI 指标分布')
ax2.set_xlabel('CCI')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 6. 绩效对比表
comparison = {}
for k in improved_result['metrics']:
    comparison[k] = f"改进: {improved_result['metrics'][k]}  |  经典: {classic_result['metrics'][k]}"
plot_metrics_table(comparison, title='改进 vs 经典布林带')

## 结果讨论

### 策略表现

1. **EMA 优势**: 比 SMA 更快响应价格变化，减少信号滞后
2. **ATR 自适应**: 高波动期自动扩大带宽，减少假突破信号
3. **ADX 过滤**: 在趋势/震荡市场分别采用不同策略逻辑
4. **多重确认**: MACD + CCI 减少单一指标的误判

### 局限性

- 日线数据无法复现论文中基于分钟线的高频结果
- 参数 (ADX 阈值, ATR 乘数) 可能需要针对不同市场环境调整
- 纯规则策略在非典型市场环境下可能失效

### 改进方向

- 使用分钟线数据提升信号频率
- 加入自适应参数调整机制
- 结合成交量确认突破信号的有效性
- 加入止损/止盈规则完善风控